In [ ]:
from __future__ import annotations
import argparse
import csv
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from Datasets.data.processed.burg_gen.burg_gen import solve_burgers
from prog import hlprs
from prog.hlprs import savefig_atomic
from prog.mlps import SirenMLP, SimpleMLP
from utils.derivative_utils import (
    autograd_spatial_derivatives,
    compute_error_metrics,
    fd_first_periodic,
    fd_second_periodic,
    fd_third_periodic,
)
from utils.extract_pde_ls import extract_pde_ls
from utils.fit_utils import fit_model_to_data
from utils.tv_utils import available_tv_types, dispatch_tv


In [ ]:
DEFAULT_LAMBDAS = [0.0, 1e-8, 1e-6, 1e-4, 1e-2]

Generate Datasets and normalize coordinates

In [ ]:
import utils.data_prep_utils as dpu

x_grid, u_fin, t_end, (t_grid, u_grid) = solve_burgers(
    N=int(cfg.burgers_N),
    L=float(cfg.burgers_L),
    nu=float(cfg.burgers_nu),
    dt=float(cfg.burgers_dt),
    T=float(cfg.burgers_T),
    seed=int(cfg.seed),
    return_history=True,
)

t_train,x_train,y_clean,y_noise=dpu.PDETrainDataset(
    t_grid=t_grid, 
    x_grid=x_grid, 
    u_grid=u_grid,
    stride_t=cfg.stride_t, 
    stride_x=cfg.stride_x, 
    noise_level=cfg.noise_level,
    seed=cfg.seed,
    normalize=cfg.normalize
)

In [ ]:
# Set up TV regularization (specific to this tv experiment)
tv_fn = dispatch_tv(cfg.tv_type)
tv_terms = [float(cfg.tv_lambda),tv_fn]

In [ ]:
# Fit the model to the data
base_model, hist = fit_model_to_data(
    base_model,
    t_train,
    x_train,
    y_noise,
    epochs=int(cfg.epochs),
    batch_size=int(cfg.batch_size),
    lr=float(cfg.lr),
    device=str(cfg.device),
    log_every=max(1, int(cfg.epochs) // 10),
    tv_terms=tv_terms,
)

In [ ]:
history_rows = []
if hist.rows:
    for r in hist.rows:
        history_rows.append(
            {
                "epoch": int(r.epoch),
                "total_loss": float(r.total_loss),
                "data_loss": float(r.data_loss),
                "pde_loss": float(r.pde_loss),
                "tv_loss": float(r.tv_loss),
            }
        )
else:
    for i, l in enumerate(hist.losses):
        history_rows.append({"epoch": int(i), "total_loss": float(l), "data_loss": float("nan"), "pde_loss": 0.0, "tv_loss": float("nan")})

_write_csv(run_dir / "history.csv", history_rows, ["epoch", "total_loss", "data_loss", "pde_loss", "tv_loss"])
